# ERCOT Load + Weather Data Combiner

This notebook:
1. Loads all 8 yearly ERCOT Native Load Excel files (2018–2025)
2. Fixes two known quirks in the raw data
3. Stacks them into one combined load DataFrame
4. Loads the weather CSV you downloaded earlier
5. Merges everything into one final dataset ready for modeling

### Known quirks we handle:
- **Column name inconsistency**: 2018–2020 use `HourEnding`, 2021–2025 use `Hour Ending`
- **24:00 timestamps**: ERCOT marks end-of-day as `12/31/2018 24:00` — pandas can't parse this, so we convert it to the next day at `00:00`

## Step 1: Imports

In [1]:
import pandas as pd
import numpy as np
import os

## Step 2: Define File Paths

Update `DATA_FOLDER` to wherever you saved the ERCOT Excel files.
Update `WEATHER_FILE` to wherever you saved the weather CSV from the download notebook.

In [3]:
# ── UPDATE THESE PATHS ──────────────────────────────────────────────────────
DATA_FOLDER  = "/Users/bestek/PycharmProjects/ercot-electricity-demand-forecasting/raw_data"   # folder containing the ERCOT Excel files
WEATHER_FILE = "/Users/bestek/PycharmProjects/ercot-electricity-demand-forecasting/processed_data/ercot_weather_wide_clean.csv"  # from the weather download notebook
# ────────────────────────────────────────────────────────────────────────────

LOAD_FILES = {
    2017: os.path.join(DATA_FOLDER, "Native_Load_2017.xlsx"),
    2018: os.path.join(DATA_FOLDER, "Native_Load_2018.xlsx"),
    2019: os.path.join(DATA_FOLDER, "Native_Load_2019.xlsx"),
    2020: os.path.join(DATA_FOLDER, "Native_Load_2020.xlsx"),
    2021: os.path.join(DATA_FOLDER, "Native_Load_2021.xlsx"),
    2022: os.path.join(DATA_FOLDER, "Native_Load_2022.xlsx"),
    2023: os.path.join(DATA_FOLDER, "Native_Load_2023.xlsx"),
    2024: os.path.join(DATA_FOLDER, "Native_Load_2024.xlsx"),
    2025: os.path.join(DATA_FOLDER, "Native_Load_2025.xlsx"),
}

# Verify files exist
for year, path in LOAD_FILES.items():
    status = "✓" if os.path.exists(path) else "✗ NOT FOUND"
    print(f"  {year}: {status}")

  2017: ✓
  2018: ✓
  2019: ✓
  2020: ✓
  2021: ✓
  2022: ✓
  2023: ✓
  2024: ✓
  2025: ✓


## Step 3: The Datetime Fix

ERCOT uses `24:00` to represent the end of a day (e.g., `01/01/2018 24:00`).
This is technically valid in some energy standards but pandas cannot parse it.

**The fix:** replace `24:00` with `00:00` and add 1 day, so:
- `01/01/2018 24:00` → `01/02/2018 00:00` ✓

This correctly represents midnight at the boundary between two days.

In [4]:
def fix_hour_ending(raw_string):
    """
    Convert ERCOT HourEnding string to a proper pandas Timestamp.
    Handles two special cases:
      - '24:00'     → end of day, shift to next day 00:00
      - '02:00 DST' → fall-back hour, shift forward 1 hour to 03:00
                      so it doesn't collide with the real 02:00
    """
    s = str(raw_string).strip()
    
    if "24:00" in s:
        s = s.replace("24:00", "00:00")
        return pd.Timestamp(s) + pd.Timedelta(days=1)
    elif "DST" in s:
        # Remove the DST label, parse, then shift forward 1 hour
        # e.g. '11/04/2018 02:00 DST' → 2018-11-04 03:00:00
        s = s.replace(" DST", "")
        return pd.Timestamp(s) + pd.Timedelta(hours=1)
    else:
        return pd.Timestamp(s)

# Quick tests
print(fix_hour_ending("01/01/2018 24:00"))       # → 2018-01-02 00:00:00
print(fix_hour_ending("11/04/2018 02:00 DST"))   # → 2018-11-04 03:00:00  (no duplicate!)
print(fix_hour_ending("01/01/2018 01:00"))        # → 2018-01-01 01:00:00

2018-01-02 00:00:00
2018-11-04 03:00:00
2018-01-01 01:00:00


## Step 4: Load & Stack All ERCOT Files

In [5]:
ZONES = ["COAST", "EAST", "FWEST", "NORTH", "NCENT", "SOUTH", "SCENT", "WEST"]

yearly_dfs = []

for year, path in LOAD_FILES.items():
    print(f"Loading {year}...", end=" ")
    
    df = pd.read_excel(path)
    
    # Step 1: Normalize the column name (handles 'HourEnding' vs 'Hour Ending')
    df.columns = [c.strip().replace(" ", "") for c in df.columns]
    # Now it's always 'HourEnding' regardless of year
    
    # Step 2: Fix the datetime
    df["datetime"] = df["HourEnding"].apply(fix_hour_ending)
    
    # Step 3: Keep only what we need
    df = df[["datetime"] + ZONES + ["ERCOT"]]
    
    # Step 4: Rename zone columns to make it clear these are load (MW) values
    rename_map = {zone: f"{zone}_load_mw" for zone in ZONES}
    rename_map["ERCOT"] = "ERCOT_total_load_mw"
    df = df.rename(columns=rename_map)
    
    yearly_dfs.append(df)
    print(f"Done! {len(df)} rows")

# Stack all years into one DataFrame
df_load = pd.concat(yearly_dfs, ignore_index=True)
df_load = df_load.sort_values("datetime").reset_index(drop=True)

print(f"\nCombined load data: {df_load.shape}")
print(f"Date range: {df_load['datetime'].min()} → {df_load['datetime'].max()}")

Loading 2017... Done! 8760 rows
Loading 2018... Done! 8760 rows
Loading 2019... Done! 8760 rows
Loading 2020... Done! 8784 rows
Loading 2021... Done! 8760 rows
Loading 2022... Done! 8760 rows
Loading 2023... Done! 8760 rows
Loading 2024... Done! 8784 rows
Loading 2025... Done! 8760 rows

Combined load data: (78888, 10)
Date range: 2017-01-01 01:00:00 → 2026-01-01 00:00:00


## Step 5: Quick Sanity Check on Load Data

In [6]:
print("=== MISSING VALUES ===")
print(df_load.isnull().sum())

print("\n=== DUPLICATE TIMESTAMPS ===")
dupes = df_load[df_load.duplicated(subset='datetime', keep=False)]
print(f"Duplicate rows: {len(dupes)}")
if len(dupes) > 0:
    print(dupes[["datetime", "ERCOT_total_load_mw"]].head(10))

print("\n=== SAMPLE DATA ===")
df_load.head()

=== MISSING VALUES ===
datetime               0
COAST_load_mw          0
EAST_load_mw           0
FWEST_load_mw          0
NORTH_load_mw          0
NCENT_load_mw          0
SOUTH_load_mw          0
SCENT_load_mw          0
WEST_load_mw           0
ERCOT_total_load_mw    0
dtype: int64

=== DUPLICATE TIMESTAMPS ===
Duplicate rows: 18
                 datetime  ERCOT_total_load_mw
7393  2017-11-05 03:00:00         32235.959803
7394  2017-11-05 03:00:00         32922.363231
16129 2018-11-04 03:00:00         28555.931880
16130 2018-11-04 03:00:00         28135.607678
24865 2019-11-03 03:00:00         33309.960879
24866 2019-11-03 03:00:00         33513.366296
33601 2020-11-01 03:00:00         30232.558552
33602 2020-11-01 03:00:00         30005.218299
42505 2021-11-07 03:00:00         32661.760943
42506 2021-11-07 03:00:00         32503.665084

=== SAMPLE DATA ===


,datetime,COAST_load_mw,EAST_load_mw,FWEST_load_mw,NORTH_load_mw,NCENT_load_mw,SOUTH_load_mw,SCENT_load_mw,WEST_load_mw,ERCOT_total_load_mw
0,2017-01-01 01:00:00,8791.789509,896.746302,1997.717635,683.621986,9239.153285,2366.632745,4490.781365,954.192864,29420.635691
1,2017-01-01 02:00:00,8569.708419,865.930568,1997.781319,677.969375,9104.997245,2332.744630,4370.656830,951.025166,28870.813551
2,2017-01-01 03:00:00,8326.425638,839.051175,1993.699160,671.998949,8988.035201,2237.506202,4210.650003,944.357749,28211.724078
3,2017-01-01 04:00:00,8137.497400,822.829332,1995.540876,675.267971,8979.148462,2178.102265,4088.713039,943.188703,27820.288047
4,2017-01-01 05:00:00,8011.869581,814.016188,1995.253501,663.619875,9033.547636,2133.953870,4021.757095,954.937932,27628.955677


In [7]:
before = len(df_load)
df_load = df_load.drop_duplicates(subset='datetime', keep='first').reset_index(drop=True)
print(f'\nDropped {before - len(df_load)} rows → {len(df_load):,} rows remaining')


Dropped 9 rows → 78,879 rows remaining


In [8]:
df_load = df_load.set_index('datetime')

full_index = pd.date_range(start=df_load.index.min(), end=df_load.index.max(), freq='h')
df_load = df_load.reindex(full_index)

print(f'Rows after reindex : {len(df_load):,}')
print(f'NaN rows inserted  : {df_load.isnull().any(axis=1).sum()}')

print('\n2018-03-11 around 2AM (before interpolation):')
display(df_load.loc['2018-03-11 00:00':'2018-03-11 04:00'])

Rows after reindex : 78,888
NaN rows inserted  : 9

2018-03-11 around 2AM (before interpolation):


,COAST_load_mw,EAST_load_mw,FWEST_load_mw,NORTH_load_mw,NCENT_load_mw,SOUTH_load_mw,SCENT_load_mw,WEST_load_mw,ERCOT_total_load_mw
2018-03-11 00:00:00,10354.996773,1129.322809,2415.344920,625.532210,9972.589637,3070.801993,5355.713366,951.953383,33876.255091
2018-03-11 01:00:00,9878.107417,1074.000272,2365.666194,594.102749,9295.332863,2900.649516,4942.708186,917.637862,31968.205059
2018-03-11 02:00:00,9540.806925,1034.490366,2344.170560,581.447655,8782.605268,2761.624620,4640.516727,898.567500,30584.229620
2018-03-11 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-03-11 04:00:00,9382.120166,1016.153058,2361.398130,577.619915,8474.674743,2681.769065,4449.719455,900.157764,29843.612296


In [9]:
numeric_cols = df_load.select_dtypes(include='number').columns.tolist()
df_load[numeric_cols] = df_load[numeric_cols].interpolate(method='linear')

print('2018-03-11 around 2AM (after interpolation):')
display(df_load.loc['2018-03-11 00:00':'2018-03-11 04:00'])

print(f'Rows after : {len(df_load):,}')

2018-03-11 around 2AM (after interpolation):


,COAST_load_mw,EAST_load_mw,FWEST_load_mw,NORTH_load_mw,NCENT_load_mw,SOUTH_load_mw,SCENT_load_mw,WEST_load_mw,ERCOT_total_load_mw
2018-03-11 00:00:00,10354.996773,1129.322809,2415.344920,625.532210,9972.589637,3070.801993,5355.713366,951.953383,33876.255091
2018-03-11 01:00:00,9878.107417,1074.000272,2365.666194,594.102749,9295.332863,2900.649516,4942.708186,917.637862,31968.205059
2018-03-11 02:00:00,9540.806925,1034.490366,2344.170560,581.447655,8782.605268,2761.624620,4640.516727,898.567500,30584.229620
2018-03-11 03:00:00,9461.463546,1025.321712,2352.784345,579.533785,8628.640005,2721.696843,4545.118091,899.362632,30213.920958
2018-03-11 04:00:00,9382.120166,1016.153058,2361.398130,577.619915,8474.674743,2681.769065,4449.719455,900.157764,29843.612296


Rows after : 78,888


In [10]:
# Save the DatetimeIndex BEFORE reset_index
new_datetime = pd.DatetimeIndex(df_load.index)  # explicitly cast to be safe
df_load = df_load.reset_index(drop=True)

df_load['datetime']     = new_datetime.strftime('%Y-%m-%d %H:%M:%S')
df_load['hour']         = new_datetime.hour
df_load['day_of_week']  = new_datetime.dayofweek
df_load['month']        = new_datetime.month
df_load['year']         = new_datetime.year
df_load['is_weekend']   = (new_datetime.dayofweek >= 5).astype(int)

In [11]:
print(f'Rows after reindex : {len(df_load):,}')
print(f'NaN rows inserted  : {df_load.isnull().any(axis=1).sum()}')

Rows after reindex : 78,888
NaN rows inserted  : 0


In [15]:
df_load = df_load[df_load['year'] != 2017]
df_load

,COAST_load_mw,EAST_load_mw,FWEST_load_mw,NORTH_load_mw,NCENT_load_mw,SOUTH_load_mw,SCENT_load_mw,WEST_load_mw,ERCOT_total_load_mw,datetime,hour,day_of_week,month,year,is_weekend
8759,11452.163689,1834.871741,2856.941683,1145.645045,18944.980941,3726.039934,9172.714892,1701.543039,50834.900964,2018-01-01 00:00:00,0,0,1,2018,0
8760,11425.979115,1852.663959,2823.409245,1135.360907,18584.343617,3831.649454,9151.190703,1762.472684,50567.069682,2018-01-01 01:00:00,1,0,1,2018,0
8761,11408.418023,1850.169452,2809.745403,1136.630855,18524.141392,3988.271046,9144.993712,1754.718094,50617.087977,2018-01-01 02:00:00,2,0,1,2018,0
8762,11405.198365,1858.269586,2797.802576,1135.930264,18532.056616,4076.086451,9141.036615,1747.919615,50694.300087,2018-01-01 03:00:00,3,0,1,2018,0
8763,11450.560138,1879.623596,2807.793880,1146.069491,18647.444612,4154.939804,9157.956866,1755.203307,50999.591693,2018-01-01 04:00:00,4,0,1,2018,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78883,12400.963382,1727.552125,7809.048752,1695.381416,13247.718927,3896.191878,7725.023700,1366.821485,49868.701664,2025-12-31 20:00:00,20,2,12,2025,0
78884,12179.429071,1730.405940,7914.007494,1680.087796,12946.659518,3743.168715,7514.969312,1361.081598,49069.809445,2025-12-31 21:00:00,21,2,12,2025,0
78885,12004.088675,1710.928961,7810.496116,1659.990999,12678.814342,3648.419252,7381.238920,1363.000085,48256.977350,2025-12-31 22:00:00,22,2,12,2025,0
78886,11792.978893,1678.576949,7734.808047,1594.513759,12359.687434,3619.904628,7258.624248,1355.590206,47394.684164,2025-12-31 23:00:00,23,2,12,2025,0


In [20]:
df_load['datetime'] = pd.to_datetime(df_load['datetime'], format='%Y-%m-%d %H:%M:%S')

/var/folders/3z/gkmtk37953l3ph4v50b84b300000gn/T/ipykernel_31497/3298009827.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_load['datetime'] = pd.to_datetime(df_load['datetime'], format='%Y-%m-%d %H:%M:%S')


## Step 6: Load Weather Data

In [12]:
df_weather = pd.read_csv(WEATHER_FILE, parse_dates=["datetime"])

print("Raw datetime sample:", df_weather['datetime'].iloc[0])
print("Raw datetime dtype: ", df_weather['datetime'].dtype)


print(f"\nParsed datetime sample: {df_weather['datetime'].iloc[0]}")
print(f"Parsed datetime dtype:  {df_weather['datetime'].dtype}")
print(f"\nWeather data shape: {df_weather.shape}")
print(f"Date range: {df_weather['datetime'].min()} → {df_weather['datetime'].max()}")
print(f"\nWeather columns: {df_weather.columns.tolist()}")

Raw datetime sample: 2018-01-01 00:00:00
Raw datetime dtype:  datetime64[ns]

Parsed datetime sample: 2018-01-01 00:00:00
Parsed datetime dtype:  datetime64[ns]

Weather data shape: (70128, 39)
Date range: 2018-01-01 00:00:00 → 2025-12-31 23:00:00

Weather columns: ['datetime', 'COAST_temperature_f', 'EAST_temperature_f', 'FWEST_temperature_f', 'NCENT_temperature_f', 'NORTH_temperature_f', 'SCENT_temperature_f', 'SOUTH_temperature_f', 'WEST_temperature_f', 'COAST_relative_humidity_pct', 'EAST_relative_humidity_pct', 'FWEST_relative_humidity_pct', 'NCENT_relative_humidity_pct', 'NORTH_relative_humidity_pct', 'SCENT_relative_humidity_pct', 'SOUTH_relative_humidity_pct', 'WEST_relative_humidity_pct', 'COAST_wind_speed_mph', 'EAST_wind_speed_mph', 'FWEST_wind_speed_mph', 'NCENT_wind_speed_mph', 'NORTH_wind_speed_mph', 'SCENT_wind_speed_mph', 'SOUTH_wind_speed_mph', 'WEST_wind_speed_mph', 'COAST_solar_radiation_wm2', 'EAST_solar_radiation_wm2', 'FWEST_solar_radiation_wm2', 'NCENT_solar_radi

## Step 7: Merge Load + Weather

We do a **left join** on `datetime` — this keeps all ERCOT rows and attaches weather data where available.
Any ERCOT hours without a matching weather timestamp will show `NaN` in weather columns (we'll check for these).

In [22]:
df_merged = df_load.merge(df_weather, on="datetime", how="left")

print(f"Merged dataset shape: {df_merged.shape}")
print(f"Rows: {len(df_merged):,}")
print(f"Columns: {len(df_merged.columns)}")

# Check how many rows didn't get weather data
missing_weather = df_merged["COAST_temperature_f"].isnull().sum()
print(f"\nRows missing weather data: {missing_weather} ({missing_weather/len(df_merged)*100:.2f}%)")

Merged dataset shape: (70129, 53)
Rows: 70,129
Columns: 53

Rows missing weather data: 1 (0.00%)


In [25]:
import pandas as pd

def check_and_clean_time_columns(df):
    """
    1. Checks whether _x and _y versions of time columns are identical.
    2. If identical: drops the _y version, renames _x → clean name.
    3. If NOT identical: reports which rows differ so you can investigate.
    4. Adds a 'is_dst_change_day' column: 1 if that date is a spring-forward
       or fall-back day, 0 otherwise.
    """

    time_cols = ['hour', 'day_of_week', 'month', 'year', 'is_weekend']

    print('=== TIME COLUMN COMPARISON ===')
    all_match = True

    for col in time_cols:
        x_col = f'{col}_x'
        y_col = f'{col}_y'

        # Skip if one or both don't exist
        if x_col not in df.columns and y_col not in df.columns:
            print(f'{col:15s} → neither _x nor _y found, skipping')
            continue
        if x_col not in df.columns:
            print(f'{col:15s} → only _y exists, renaming to {col}')
            df = df.rename(columns={y_col: col})
            continue
        if y_col not in df.columns:
            print(f'{col:15s} → only _x exists, renaming to {col}')
            df = df.rename(columns={x_col: col})
            continue

        # Both exist — compare them
        mismatches = (df[x_col] != df[y_col]).sum()

        if mismatches == 0:
            print(f'{col:15s} → ✅ identical ({len(df):,} rows match) — keeping one, dropping the other')
            df = df.drop(columns=[y_col])
            df = df.rename(columns={x_col: col})
        else:
            all_match = False
            print(f'{col:15s} → ⚠️  {mismatches:,} mismatches found!')
            print(f'  Sample mismatched rows:')
            bad_rows = df[df[x_col] != df[y_col]][['datetime', x_col, y_col]].head(5)
            print(bad_rows.to_string(index=False))
            # Still keep _x version (from load side) and drop _y
            df = df.drop(columns=[y_col])
            df = df.rename(columns={x_col: col})

    if all_match:
        print('\n✅ All time columns were identical — cleaned up successfully.')
    else:
        print('\n⚠️  Some columns had mismatches — kept _x version, investigate the rows above.')

    # === Add DST change day flag ===
    # DST in the US (Texas/ERCOT):
    #   Spring forward: 2nd Sunday of March   (clocks go +1hr at 2AM)
    #   Fall back:      1st Sunday of November (clocks go -1hr at 2AM)

    print('\n=== ADDING is_dst_change_day COLUMN ===')

    dt = pd.to_datetime(df['datetime'])

    def is_dst_day(date_series):
        flags = []
        for dt_val in date_series:
            m = dt_val.month
            d = dt_val.day
            dow = dt_val.dayofweek  # 0=Mon, 6=Sun

            is_dst = False

            # Spring forward: 2nd Sunday of March
            if m == 3 and dow == 6:  # Sunday in March
                # 2nd Sunday = day >= 8 and <= 14
                if 8 <= d <= 14:
                    is_dst = True

            # Fall back: 1st Sunday of November
            if m == 11 and dow == 6:  # Sunday in November
                # 1st Sunday = day >= 1 and <= 7
                if 1 <= d <= 7:
                    is_dst = True

            flags.append(int(is_dst))
        return flags

    df['is_dst_change_day'] = is_dst_day(dt)

    dst_days = df[df['is_dst_change_day'] == 1]['datetime'].dt.strftime('%Y-%m-%d').unique()
    print(f'DST change days flagged: {len(dst_days)}')
    print(dst_days)

    return df


# --- Run it ---
df_merged = check_and_clean_time_columns(df_merged)

print(f'\nFinal shape: {df_merged.shape}')
print(f'Columns: {df_merged.columns.tolist()}')
df_merged.head(3)

=== TIME COLUMN COMPARISON ===
hour            → ⚠️  1 mismatches found!
  Sample mismatched rows:
  datetime  hour_x  hour_y
2026-01-01       0     NaN
day_of_week     → ⚠️  1 mismatches found!
  Sample mismatched rows:
  datetime  day_of_week_x  day_of_week_y
2026-01-01              3            NaN
month           → ⚠️  1 mismatches found!
  Sample mismatched rows:
  datetime  month_x  month_y
2026-01-01        1      NaN
year            → ⚠️  1 mismatches found!
  Sample mismatched rows:
  datetime  year_x  year_y
2026-01-01    2026     NaN
is_weekend      → ⚠️  1 mismatches found!
  Sample mismatched rows:
  datetime  is_weekend_x  is_weekend_y
2026-01-01             0           NaN

⚠️  Some columns had mismatches — kept _x version, investigate the rows above.

=== ADDING is_dst_change_day COLUMN ===
DST change days flagged: 16
['2018-03-11' '2018-11-04' '2019-03-10' '2019-11-03' '2020-03-08'
 '2020-11-01' '2021-03-14' '2021-11-07' '2022-03-13' '2022-11-06'
 '2023-03-12' '2023-11

,COAST_load_mw,EAST_load_mw,FWEST_load_mw,NORTH_load_mw,NCENT_load_mw,SOUTH_load_mw,SCENT_load_mw,WEST_load_mw,ERCOT_total_load_mw,datetime,...,COAST_solar_radiation_wm2,EAST_solar_radiation_wm2,FWEST_solar_radiation_wm2,NCENT_solar_radiation_wm2,NORTH_solar_radiation_wm2,SCENT_solar_radiation_wm2,SOUTH_solar_radiation_wm2,WEST_solar_radiation_wm2,is_holiday,is_dst_change_day
0,11452.163689,1834.871741,2856.941683,1145.645045,18944.980941,3726.039934,9172.714892,1701.543039,50834.900964,2018-01-01 00:00:00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
1,11425.979115,1852.663959,2823.409245,1135.360907,18584.343617,3831.649454,9151.190703,1762.472684,50567.069682,2018-01-01 01:00:00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
2,11408.418023,1850.169452,2809.745403,1136.630855,18524.141392,3988.271046,9144.993712,1754.718094,50617.087977,2018-01-01 02:00:00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0


## Step 8: Final Column Overview

In [26]:
print("ALL COLUMNS IN FINAL DATASET:")
print()

load_cols    = [c for c in df_merged.columns if 'load' in c]
temp_cols    = [c for c in df_merged.columns if 'temperature' in c]
wind_cols    = [c for c in df_merged.columns if 'wind' in c]
solar_cols   = [c for c in df_merged.columns if 'radiation' in c or 'irradiance' in c]
cal_cols     = [c for c in df_merged.columns if c in ['hour','day_of_week','month','year','is_weekend','is_holiday']]
humidity_cols= [c for c in df_merged.columns if 'humidity' in c]

print(f"📊 Load columns ({len(load_cols)}):    {load_cols}")
print(f"🌡️  Temperature ({len(temp_cols)}):    {temp_cols}")
print(f"💧 Humidity ({len(humidity_cols)}):       {humidity_cols}")
print(f"💨 Wind ({len(wind_cols)}):             {wind_cols}")
print(f"☀️  Solar ({len(solar_cols)}):           {solar_cols}")
print(f"📅 Calendar ({len(cal_cols)}):         {cal_cols}")

ALL COLUMNS IN FINAL DATASET:

📊 Load columns (9):    ['COAST_load_mw', 'EAST_load_mw', 'FWEST_load_mw', 'NORTH_load_mw', 'NCENT_load_mw', 'SOUTH_load_mw', 'SCENT_load_mw', 'WEST_load_mw', 'ERCOT_total_load_mw']
🌡️  Temperature (8):    ['COAST_temperature_f', 'EAST_temperature_f', 'FWEST_temperature_f', 'NCENT_temperature_f', 'NORTH_temperature_f', 'SCENT_temperature_f', 'SOUTH_temperature_f', 'WEST_temperature_f']
💧 Humidity (8):       ['COAST_relative_humidity_pct', 'EAST_relative_humidity_pct', 'FWEST_relative_humidity_pct', 'NCENT_relative_humidity_pct', 'NORTH_relative_humidity_pct', 'SCENT_relative_humidity_pct', 'SOUTH_relative_humidity_pct', 'WEST_relative_humidity_pct']
💨 Wind (8):             ['COAST_wind_speed_mph', 'EAST_wind_speed_mph', 'FWEST_wind_speed_mph', 'NCENT_wind_speed_mph', 'NORTH_wind_speed_mph', 'SCENT_wind_speed_mph', 'SOUTH_wind_speed_mph', 'WEST_wind_speed_mph']
☀️  Solar (8):           ['COAST_solar_radiation_wm2', 'EAST_solar_radiation_wm2', 'FWEST_solar_r

## Step 9: Preview Final Dataset

In [27]:
# Show a few key columns to confirm the merge looks right
preview_cols = [
    "datetime", 
    "ERCOT_total_load_mw",
    "NCENT_load_mw",         # Dallas — largest zone
    "NCENT_temperature_f",   # matching weather
    "NCENT_wind_speed_mph",
    "NCENT_solar_radiation_wm2",
    "hour", "day_of_week", "is_holiday"
]

df_merged[preview_cols].head(24)  # first 24 hours

,datetime,ERCOT_total_load_mw,NCENT_load_mw,NCENT_temperature_f,NCENT_wind_speed_mph,NCENT_solar_radiation_wm2,hour,day_of_week,is_holiday
0,2018-01-01 00:00:00,50834.900964,18944.980941,21.290000,14.306311,0.0,0,0,1.0
1,2018-01-01 01:00:00,50567.069682,18584.343617,20.480000,14.085997,0.0,1,0,1.0
2,2018-01-01 02:00:00,50617.087977,18524.141392,19.850000,13.350974,0.0,2,0,1.0
3,2018-01-01 03:00:00,50694.300087,18532.056616,19.220001,13.425728,0.0,3,0,1.0
4,2018-01-01 04:00:00,50999.591693,18647.444612,18.500000,12.432971,0.0,4,0,1.0
5,2018-01-01 05:00:00,51723.732017,19002.102222,18.230000,12.432971,0.0,5,0,1.0
6,2018-01-01 06:00:00,52955.634255,19477.360921,16.340000,11.548207,0.0,6,0,1.0
7,2018-01-01 07:00:00,54303.082184,19984.222251,16.160000,10.885712,0.0,7,0,1.0
8,2018-01-01 08:00:00,55099.267435,20289.365819,16.430000,11.340508,10.0,8,0,1.0
9,2018-01-01 09:00:00,55553.887208,20338.611561,18.320000,12.384578,138.0,9,0,1.0


---
# Feature Engineering

Now that the data is clean, we engineer new features. 
These are derived columns that help the models learn patterns 
that aren't obvious from raw values alone.

We'll add features in 4 groups:
1. **Cyclic time features** — encode hour/month as sine/cosine waves
2. **Lag features** — yesterday's load, last week's load
3. **Rolling averages** — smoothed recent load trends
4. **Heat index** — combines temperature + humidity into one comfort metric

## Step 1: Cyclic Time Encoding

**Why this matters:** Hour 23 and hour 0 are only 1 hour apart, but numerically 
they're 23 apart. A model treating hour as a plain number gets confused — it 
thinks midnight and 11 PM are very different when they're actually neighbors.

**The fix:** encode hour (and month) as a point on a circle using sine and cosine. 
This way hour 23 and hour 0 end up close together in the model's feature space.

```
hour_sin = sin(2π × hour / 24)
hour_cos = cos(2π × hour / 24)
```

In [30]:
# Hour of day (cycles every 24 hours)
df_merged['hour_sin'] = np.sin(2 * np.pi * df_merged['hour'] / 24)
df_merged['hour_cos'] = np.cos(2 * np.pi * df_merged['hour'] / 24)

# Month of year (cycles every 12 months)
df_merged['month_sin'] = np.sin(2 * np.pi * df_merged['month'] / 12)
df_merged['month_cos'] = np.cos(2 * np.pi * df_merged['month'] / 12)

# Day of week (cycles every 7 days)
df_merged['dow_sin'] = np.sin(2 * np.pi * df_merged['day_of_week'] / 7)
df_merged['dow_cos'] = np.cos(2 * np.pi * df_merged['day_of_week'] / 7)

print("Cyclic features added: hour_sin, hour_cos, month_sin, month_cos, dow_sin, dow_cos")
print(df_merged[['hour','hour_sin','hour_cos']].iloc[:5].round(3))

Cyclic features added: hour_sin, hour_cos, month_sin, month_cos, dow_sin, dow_cos
   hour  hour_sin  hour_cos
0     0     0.000     1.000
1     1     0.259     0.966
2     2     0.500     0.866
3     3     0.707     0.707
4     4     0.866     0.500


## Step 2: Lag Features

**Why this matters:** Electricity demand right now is heavily influenced by 
what it was 24 hours ago (same hour yesterday) and 168 hours ago (same hour 
last week). These are called **lag features** — we're giving the model 
a direct memory of recent history.

We create lags on `ERCOT_total_load_mw` as the main target, 
and also on `NCENT_temperature_f` as the most influential weather variable.

In [31]:
# Sort by time first — critical for lag features to be correct
df_merged = df_merged.sort_values('datetime').reset_index(drop=True)

# Load lags
df_merged['load_lag_24h']  = df_merged['ERCOT_total_load_mw'].shift(24)   # same hour yesterday
df_merged['load_lag_48h']  = df_merged['ERCOT_total_load_mw'].shift(48)   # same hour 2 days ago
df_merged['load_lag_168h'] = df_merged['ERCOT_total_load_mw'].shift(168)  # same hour last week

# Temperature lag for ERCOT-wide proxy (using NCENT = Dallas, largest zone)
df_merged['temp_lag_24h']  = df_merged['NCENT_temperature_f'].shift(24)

print("Lag features added!")
print(f"NaNs introduced by lags (expected — first 168 rows): {df_merged['load_lag_168h'].isnull().sum()}")
df_merged[['datetime','ERCOT_total_load_mw','load_lag_24h','load_lag_168h']].iloc[165:172]

Lag features added!
NaNs introduced by lags (expected — first 168 rows): 168


,datetime,ERCOT_total_load_mw,load_lag_24h,load_lag_168h
165,2018-01-07 21:00:00,37102.952884,37489.805354,NaN
166,2018-01-07 22:00:00,35880.205093,36921.360889,NaN
167,2018-01-07 23:00:00,33992.684812,35772.332555,NaN
168,2018-01-08 00:00:00,32063.404932,34314.168708,50834.900964
169,2018-01-08 01:00:00,30767.007132,33025.925584,50567.069682
170,2018-01-08 02:00:00,30231.165136,32160.065962,50617.087977
171,2018-01-08 03:00:00,30174.984050,31691.973813,50694.300087


## Step 3: Rolling Average Features

**Why this matters:** A rolling average smooths out noise and captures 
the recent trend. For example, a 24-hour rolling average tells the model 
whether we're in a generally high-demand period (summer heat wave) 
or a low-demand period (mild spring).

`min_periods=1` means we start computing even before we have a full window — 
useful for the beginning of the dataset.

In [32]:
df_merged['load_rolling_24h_mean'] = (
    df_merged['ERCOT_total_load_mw']
    .shift(1)                          # shift by 1 so we don't include current hour
    .rolling(window=24, min_periods=1)
    .mean()
)

df_merged['load_rolling_168h_mean'] = (
    df_merged['ERCOT_total_load_mw']
    .shift(1)
    .rolling(window=168, min_periods=1)
    .mean()
)

print("Rolling features added!")
df_merged[['datetime','ERCOT_total_load_mw','load_rolling_24h_mean','load_rolling_168h_mean']].iloc[20:25]

Rolling features added!


,datetime,ERCOT_total_load_mw,load_rolling_24h_mean,load_rolling_168h_mean
20,2018-01-01 20:00:00,57297.399445,53109.405133,53109.405133
21,2018-01-01 21:00:00,57246.956558,53308.833433,53308.833433
22,2018-01-01 22:00:00,56339.310019,53487.839030,53487.839030
23,2018-01-01 23:00:00,54402.041230,53611.816029,53611.816029
24,2018-01-02 00:00:00,52550.803685,53644.742079,53644.742079


## Step 4: Heat Index Feature

**Why this matters:** A 95°F day with 80% humidity feels much worse than 
a 95°F day with 30% humidity — and AC usage reflects that. The heat index 
combines temperature and humidity into one 'feels like' number that better 
predicts electricity demand than temperature alone.

We compute it for each zone using the standard Rothfusz formula 
(same one used by the National Weather Service).

In [33]:
def heat_index(T, RH):
    """
    Rothfusz heat index formula.
    T  = temperature in Fahrenheit
    RH = relative humidity in percent (0–100)
    Returns 'feels like' temperature in Fahrenheit.
    Only meaningful when T >= 80°F — below that we just return T.
    """
    HI = (
        -42.379
        + 2.04901523  * T
        + 10.14333127 * RH
        - 0.22475541  * T * RH
        - 0.00683783  * T**2
        - 0.05481717  * RH**2
        + 0.00122874  * T**2 * RH
        + 0.00085282  * T * RH**2
        - 0.00000199  * T**2 * RH**2
    )
    # Only apply when hot enough for heat index to be meaningful
    return np.where(T >= 80, HI, T)

ZONES = ['COAST','EAST','FWEST','NORTH','NCENT','SOUTH','SCENT','WEST']

for zone in ZONES:
    df_merged[f'{zone}_heat_index_f'] = heat_index(
        df_merged[f'{zone}_temperature_f'],
        df_merged[f'{zone}_relative_humidity_pct']
    )

print("Heat index features added for all 8 zones!")
# Show difference between temp and heat index on a hot day
hot_day = df_merged[df_merged['NCENT_temperature_f'] > 100].head(3)
print(hot_day[['datetime','NCENT_temperature_f','NCENT_relative_humidity_pct','NCENT_heat_index_f']].to_string())

Heat index features added for all 8 zones!
                datetime  NCENT_temperature_f  NCENT_relative_humidity_pct  NCENT_heat_index_f
3665 2018-06-02 17:00:00            100.03999                    35.844894          106.193080
4265 2018-06-27 17:00:00            100.40000                    31.498861          103.937326
4266 2018-06-27 18:00:00            100.40000                    30.427803          103.263336


## Step 5: Final Summary

In [34]:
print(f"Final dataset shape: {df_merged.shape}")
print(f"Total features: {df_merged.shape[1]}")
print()

# Group columns by type for a clean overview
groups = {
    'Target (load)':      [c for c in df_merged.columns if 'load_mw' in c],
    'Temperature':        [c for c in df_merged.columns if 'temperature' in c],
    'Heat Index':         [c for c in df_merged.columns if 'heat_index' in c],
    'Wind':               [c for c in df_merged.columns if 'wind_speed' in c],
    'Solar':              [c for c in df_merged.columns if 'solar' in c],
    'Humidity':           [c for c in df_merged.columns if 'humidity' in c],
    'Calendar (raw)':     ['hour','day_of_week','month','year','is_weekend','is_holiday'],
    'Cyclic encoding':    [c for c in df_merged.columns if any(x in c for x in ['_sin','_cos'])],
    'Lag features':       [c for c in df_merged.columns if 'lag' in c],
    'Rolling averages':   [c for c in df_merged.columns if 'rolling' in c],
}

for group, cols in groups.items():
    print(f"  {group} ({len(cols)}): {cols}")

Final dataset shape: (70129, 69)
Total features: 69

  Target (load) (9): ['COAST_load_mw', 'EAST_load_mw', 'FWEST_load_mw', 'NORTH_load_mw', 'NCENT_load_mw', 'SOUTH_load_mw', 'SCENT_load_mw', 'WEST_load_mw', 'ERCOT_total_load_mw']
  Temperature (8): ['COAST_temperature_f', 'EAST_temperature_f', 'FWEST_temperature_f', 'NCENT_temperature_f', 'NORTH_temperature_f', 'SCENT_temperature_f', 'SOUTH_temperature_f', 'WEST_temperature_f']
  Heat Index (8): ['COAST_heat_index_f', 'EAST_heat_index_f', 'FWEST_heat_index_f', 'NORTH_heat_index_f', 'NCENT_heat_index_f', 'SOUTH_heat_index_f', 'SCENT_heat_index_f', 'WEST_heat_index_f']
  Wind (8): ['COAST_wind_speed_mph', 'EAST_wind_speed_mph', 'FWEST_wind_speed_mph', 'NCENT_wind_speed_mph', 'NORTH_wind_speed_mph', 'SCENT_wind_speed_mph', 'SOUTH_wind_speed_mph', 'WEST_wind_speed_mph']
  Solar (8): ['COAST_solar_radiation_wm2', 'EAST_solar_radiation_wm2', 'FWEST_solar_radiation_wm2', 'NCENT_solar_radiation_wm2', 'NORTH_solar_radiation_wm2', 'SCENT_solar

## Step 6: Create Train / Validate / Test Splits

Based on our project design:
- **Train**: 2018–2023
- **Validate**: 2024 (tune model hyperparameters here)
- **Test**: 2025 (final evaluation — compare to actual data)

In [35]:
train = df_merged[df_merged["datetime"].dt.year <= 2023].copy()
val   = df_merged[df_merged["datetime"].dt.year == 2024].copy()
test  = df_merged[df_merged["datetime"].dt.year == 2025].copy()

print("SPLIT SUMMARY")
print(f"  Train (2018–2023): {len(train):>7,} rows  |  {train['datetime'].min().date()} → {train['datetime'].max().date()}")
print(f"  Validate (2024):   {len(val):>7,} rows  |  {val['datetime'].min().date()} → {val['datetime'].max().date()}")
print(f"  Test (2025):       {len(test):>7,} rows  |  {test['datetime'].min().date()} → {test['datetime'].max().date()}")
print(f"  Total:             {len(df_merged):>7,} rows")

SPLIT SUMMARY
  Train (2018–2023):  52,584 rows  |  2018-01-01 → 2023-12-31
  Validate (2024):     8,784 rows  |  2024-01-01 → 2024-12-31
  Test (2025):         8,760 rows  |  2025-01-01 → 2025-12-31
  Total:              70,129 rows


In [36]:
counts = df_merged.groupby(df_merged['datetime'].dt.year).size().reset_index()
counts.columns = ['year', 'row_count']
counts['expected'] = counts['year'].apply(lambda y: 8784 if y % 4 == 0 else 8760)
counts['diff'] = counts['row_count'] - counts['expected']
counts['status'] = counts['diff'].apply(lambda x: '✅' if x == 0 else f'⚠️  off by {x}')

print(counts.to_string(index=False))
print(f"\nTotal: {counts['row_count'].sum():,}")

 year  row_count  expected  diff           status
 2018       8760      8760     0                ✅
 2019       8760      8760     0                ✅
 2020       8784      8784     0                ✅
 2021       8760      8760     0                ✅
 2022       8760      8760     0                ✅
 2023       8760      8760     0                ✅
 2024       8784      8784     0                ✅
 2025       8760      8760     0                ✅
 2026          1      8760 -8759 ⚠️  off by -8759

Total: 70,129


## Step 11: Save Everything

In [37]:
# Full merged dataset
df_merged.to_csv("/Users/bestek/PycharmProjects/ercot-electricity-demand-forecasting/model_data/ercot_full_dataset_2018_2025.csv", index=False)
print(f"Saved: ercot_full_dataset_2018_2025.csv  ({df_merged.shape[0]:,} rows x {df_merged.shape[1]} cols)")

# Individual splits
train.to_csv("/Users/bestek/PycharmProjects/ercot-electricity-demand-forecasting/model_data/ercot_train_2018_2023.csv", index=False)
val.to_csv("/Users/bestek/PycharmProjects/ercot-electricity-demand-forecasting/model_data/ercot_validate_2024.csv", index=False)
test.to_csv("/Users/bestek/PycharmProjects/ercot-electricity-demand-forecasting/model_data/ercot_test_2025.csv", index=False)

print(f"Saved: ercot_train_2018_2023.csv         ({len(train):,} rows)")
print(f"Saved: ercot_validate_2024.csv           ({len(val):,} rows)")
print(f"Saved: ercot_test_2025.csv               ({len(test):,} rows)")
print("\nAll files saved!")

Saved: ercot_full_dataset_2018_2025.csv  (70,129 rows x 69 cols)
Saved: ercot_train_2018_2023.csv         (52,584 rows)
Saved: ercot_validate_2024.csv           (8,784 rows)
Saved: ercot_test_2025.csv               (8,760 rows)

All files saved!
